# ArXiv Paper Discovery and Filtering

This notebook allows you to search arXiv for papers using custom keywords and generate a text file of relevant arXiv IDs.

## Features:
- **Customizable Search**: Specify your own search terms
- **Smart Filtering**: Uses LLM-based classification to identify relevant papers
- **Multiple Search Modes**: Choose from different search strategies
- **Export Results**: Generates text files compatible with your existing data structure

## Usage:
1. Configure your search parameters in the next cell
2. Run the search and classification pipeline
3. Review results and export arXiv IDs


In [ ]:
# Import required modules
import os
import sys
from datetime import datetime

# Add the current directory to Python path to import arxiv_tools
sys.path.append('.')

# Import arxiv tools
from arxiv_tools.arxiv_discovery import ArXivDiscovery
from arxiv_tools.arxiv_filter import ArXivClassifier

print("✅ Modules imported successfully!")
print("📁 Make sure you have your .env file configured with API keys if using LLM classification")


In [ ]:
## 🔧 Configuration - Customize Your Search

# ========================================
# SEARCH CONFIGURATION - MODIFY THESE
# ========================================

# 1. SEARCH TERMS - Add your keywords here
SEARCH_TERMS = [
    'robotics AND "foundation model"',
    'robot AND "large model"', 
    'robotics AND transformer',
    'robot AND multimodal AND model'
]

# 2. ARXIV CATEGORIES - Specify which categories to search (empty list = search all)
CATEGORIES = [
    'cs.RO',  # Robotics
    'cs.AI',  # Artificial Intelligence  
    'cs.LG',  # Machine Learning
    'cs.CV',  # Computer Vision
    'cs.CL'   # Computation and Language
]

# 3. SEARCH PARAMETERS
MAX_PAPERS_PER_SEARCH = 100  # Maximum papers per search term (0 = get all available)
DATE_RANGE_MONTHS = 24       # How many months back to search (0 = all time)
SEARCH_MODE = 'balanced'     # Options: 'natural', 'balanced', 'exact', 'title_focus', 'comprehensive'

# 4. LLM CLASSIFICATION SETTINGS
USE_LLM_FILTERING = True     # Set to False to skip LLM classification
LLM_PROVIDER = 'deepseek'    # Options: 'deepseek', 'gemini'
CONFIDENCE_THRESHOLD = 0.7   # Minimum confidence for positive classification (0.0-1.0)

# 5. OUTPUT SETTINGS
OUTPUT_FILENAME = f"arxiv_ids_custom_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

# Display configuration
print("🔍 Search Configuration:")
print(f"   Search Terms: {SEARCH_TERMS}")
print(f"   Categories: {CATEGORIES if CATEGORIES else 'All categories'}")
print(f"   Max Papers per Search: {MAX_PAPERS_PER_SEARCH if MAX_PAPERS_PER_SEARCH > 0 else 'All available'}")
print(f"   Date Range: {DATE_RANGE_MONTHS} months" if DATE_RANGE_MONTHS > 0 else "   Date Range: All time")
print(f"   Search Mode: {SEARCH_MODE}")
print(f"   LLM Filtering: {'Enabled' if USE_LLM_FILTERING else 'Disabled'}")
if USE_LLM_FILTERING:
    print(f"   LLM Provider: {LLM_PROVIDER}")
    print(f"   Confidence Threshold: {CONFIDENCE_THRESHOLD}")
print(f"   Output File: {OUTPUT_FILENAME}")

print("\n💡 To modify search terms, edit the SEARCH_TERMS list above and re-run this cell.")


In [ ]:
## 🔍 Step 1: Search ArXiv for Papers

# Create configuration for ArXiv discovery
config = {
    'search_terms': SEARCH_TERMS,
    'categories': CATEGORIES,
    'max_papers_per_search': MAX_PAPERS_PER_SEARCH,
    'date_range_months': DATE_RANGE_MONTHS,
    'search_mode': SEARCH_MODE,
    'sort_by': 'relevance',
    'rate_limit_delay': 3.0
}

print("🚀 Starting ArXiv paper search...")
print("=" * 50)

# Initialize discovery system
discovery = ArXivDiscovery(config)

# Run the search
papers = discovery.run_discovery()

print(f"\n📊 Search Results:")
print(f"   Total papers found: {len(papers)}")

if papers:
    print(f"\n📄 Sample papers found:")
    for i, paper in enumerate(papers[:3], 1):
        print(f"   {i}. {paper['title'][:80]}...")
        print(f"      arXiv:{paper['arxiv_id']} | {paper['published'][:10]} | {paper['primary_category']}")
    
    if len(papers) > 3:
        print(f"   ... and {len(papers) - 3} more papers")
else:
    print("❌ No papers found. Try adjusting your search terms or expanding the date range.")


In [ ]:
## 🤖 Step 2: LLM-Based Classification (Optional)

if USE_LLM_FILTERING and papers:
    print("🧠 Starting LLM-based paper classification...")
    print("=" * 50)
    
    # Convert papers to classification format
    paper_data = []
    for paper in papers:
        paper_data.append({
            'arxiv_id': paper.get('arxiv_id'),
            'title': paper.get('title'),
            'abstract': paper.get('summary'),  # ArXivDiscovery uses 'summary' key
            'authors': paper.get('authors'),
            'published': paper.get('published'),
            'categories': paper.get('categories')
        })
    
    # Initialize classifier
    classifier = ArXivClassifier(provider=LLM_PROVIDER, rate_limit_delay=1.0)
    
    # Run classification
    classification_results = classifier.classify_papers(paper_data, confidence_threshold=CONFIDENCE_THRESHOLD)
    
    # Extract positive matches
    positive_matches = classification_results.get('positive_matches', [])
    
    print(f"\n🎯 Classification Results:")
    print(f"   Papers classified: {classification_results['classification_summary']['total_papers']}")
    print(f"   Positive matches: {len(positive_matches)}")
    print(f"   Match rate: {classification_results['classification_summary']['match_rate']:.1%}")
    print(f"   Success rate: {classification_results['classification_summary']['success_rate']:.1%}")
    
    if positive_matches:
        print(f"\n✅ Relevant papers found:")
        for i, match in enumerate(positive_matches[:5], 1):
            print(f"   {i}. {match['title'][:80]}...")
            print(f"      arXiv:{match['arxiv_id']} | Confidence: {match['confidence']:.2f}")
            if match.get('model_name'):
                print(f"      Model: {match['model_name']}")
        
        if len(positive_matches) > 5:
            print(f"   ... and {len(positive_matches) - 5} more relevant papers")
        
        # Use filtered results for export
        final_papers = positive_matches
    else:
        print("❌ No papers passed the LLM classification threshold.")
        print("💡 Consider lowering the confidence threshold or adjusting search terms.")
        final_papers = []

elif not USE_LLM_FILTERING and papers:
    print("⏭️  Skipping LLM classification (USE_LLM_FILTERING = False)")
    print("📋 Using all search results for export")
    final_papers = papers

else:
    print("⚠️  No papers to classify")
    final_papers = []

print(f"\n📈 Final Results: {len(final_papers)} papers selected for export")


In [ ]:
## 💾 Step 3: Export ArXiv IDs to Text File

if final_papers:
    print("📝 Exporting ArXiv IDs to text file...")
    print("=" * 50)
    
    # Ensure data directory exists
    os.makedirs("data/arxiv_ids", exist_ok=True)
    
    # Create full output path
    output_path = os.path.join("data/arxiv_ids", OUTPUT_FILENAME)
    
    # Extract arXiv IDs and write to file
    arxiv_ids = []
    with open(output_path, 'w', encoding='utf-8') as f:
        for paper in final_papers:
            arxiv_id = paper.get('arxiv_id')
            if arxiv_id:
                f.write(f"{arxiv_id}\n")
                arxiv_ids.append(arxiv_id)
    
    print(f"✅ Successfully exported {len(arxiv_ids)} arXiv IDs")
    print(f"📁 File saved to: {output_path}")
    
    # Show preview of exported IDs
    print(f"\n📋 Preview of exported arXiv IDs:")
    for i, arxiv_id in enumerate(arxiv_ids[:10], 1):
        print(f"   {i:2d}. {arxiv_id}")
    
    if len(arxiv_ids) > 10:
        print(f"   ... and {len(arxiv_ids) - 10} more IDs")
    
    # File statistics
    file_size = os.path.getsize(output_path)
    print(f"\n📊 File Statistics:")
    print(f"   Total IDs: {len(arxiv_ids)}")
    print(f"   File size: {file_size} bytes")
    print(f"   Format: One arXiv ID per line (compatible with existing data structure)")
    
else:
    print("❌ No papers to export. Please check your search configuration and try again.")
    print("💡 Suggestions:")
    print("   - Try broader search terms")
    print("   - Increase the date range")
    print("   - Lower the confidence threshold if using LLM filtering")
    print("   - Disable LLM filtering to export all search results")


In [ ]:
## 📊 Summary and Next Steps

print("🎉 ArXiv Discovery and Filtering Complete!")
print("=" * 50)

# Summary statistics
if 'papers' in locals() and papers:
    print(f"📈 Pipeline Summary:")
    print(f"   Initial search results: {len(papers)}")
    
    if USE_LLM_FILTERING and 'classification_results' in locals():
        print(f"   LLM classification: {classification_results['classification_summary']['success_rate']:.1%} success rate")
        print(f"   Papers passing filter: {len(positive_matches) if 'positive_matches' in locals() else 0}")
    
    if 'arxiv_ids' in locals():
        print(f"   ArXiv IDs exported: {len(arxiv_ids)}")
        print(f"   Output file: {output_path}")
    
    print(f"\n🔄 Next Steps:")
    print(f"   1. Review the exported arXiv IDs in: {OUTPUT_FILENAME}")
    print(f"   2. Use these IDs with your existing OCR and analysis pipeline")
    print(f"   3. The file format is compatible with your data/arxiv_ids/ structure")
    
    if USE_LLM_FILTERING:
        print(f"   4. Adjust confidence threshold ({CONFIDENCE_THRESHOLD}) if needed")
    
    print(f"\n💡 Tips for Better Results:")
    print(f"   - Experiment with different search modes: {['natural', 'balanced', 'exact', 'title_focus', 'comprehensive']}")
    print(f"   - Try different combinations of search terms")
    print(f"   - Adjust the date range to focus on recent or historical papers")
    print(f"   - Use category filtering to narrow down to specific domains")

else:
    print("⚠️  No results to summarize. Please run the search cells above first.")

print(f"\n🔧 To run another search:")
print(f"   1. Modify the SEARCH_TERMS in the configuration cell")
print(f"   2. Adjust other parameters as needed")
print(f"   3. Re-run all cells from the configuration onwards")


In [ ]:
## 🎛️ Quick Configuration Examples

# Uncomment and modify one of these examples, then re-run the configuration cell above

# Example 1: Search for Vision-Language Models
# SEARCH_TERMS = [
#     'vision language model',
#     'multimodal AND vision AND language',
#     'VLM AND robotics',
#     'visual instruction following'
# ]

# Example 2: Search for Transformer Architectures
# SEARCH_TERMS = [
#     'transformer AND robotics',
#     'attention mechanism AND robot',
#     'self-attention AND manipulation',
#     'transformer AND control'
# ]

# Example 3: Search for Reinforcement Learning
# SEARCH_TERMS = [
#     'reinforcement learning AND robotics',
#     'RL AND robot manipulation',
#     'policy learning AND robotics',
#     'robot AND reinforcement learning'
# ]

# Example 4: Broad AI/ML Search (no robotics focus)
# SEARCH_TERMS = [
#     'machine learning',
#     'deep learning',
#     'neural networks',
#     'artificial intelligence'
# ]
# CATEGORIES = ['cs.LG', 'cs.AI', 'stat.ML']

# Example 5: Computer Vision Focus
# SEARCH_TERMS = [
#     'computer vision',
#     'image recognition',
#     'object detection',
#     'semantic segmentation'
# ]
# CATEGORIES = ['cs.CV']

print("💡 Copy one of the examples above to the configuration cell to try different searches!")
print("🔄 Remember to re-run the configuration cell after making changes.")
